# 05 · Tarea 2 — FAIR loss

**Objetivo.** Entrenar el clasificador de impago con una **función de coste FAIR** que combina el ajuste con una **penalización de dependencia** entre la predicción y el género:

$$\mathcal{L} \;=\; \underbrace{\text{BCE}(\hat y,\ \texttt{TARGET})}_{\text{clasificación}} \;+\; \lambda\cdot\underbrace{\operatorname{D}(\hat y,\ \texttt{CODE\_GENDER})}_{\text{penalización FAIR}}$$

donde la combinación `BCE + λ·D` es el Lagrangiano del problema restringido (**D-2.2**), y `D` es una medida de dependencia **diferenciable** —HSIC con kernel RBF o `corr²`— que vale 0 solo bajo independencia (**D-2.1**, ver [`02-fair-loss.md`](../docs/teoria/02-fair-loss.md) §2 y [`00-fundamentos-dependencia.md`](../docs/teoria/00-fundamentos-dependencia.md)). La penalización opera sobre la **probabilidad `ŷ`** = $P(\texttt{TARGET}=1)$ (**D-2.6**), no sobre el logit.

**El género `s` NO es input** del modelo (X = 13 features sin género): entra **concatenado en `y_true` como `[y, s]`** y se desempaqueta dentro de la loss `y_real, s = y_true[:,0], y_true[:,1]` (**D-2.5**). Se **barre `λ`** para trazar el *trade-off* precisión↔equidad, reportando **AUC-ROC** como precisión (**D-2.4**) y **group gap** M−F como equidad (**D-2.3**; base a batir **+3,14 pp**).

El **entregable central es E5**: la tabla **modelo Base (sin FAIR) vs mejor modelo FAIR** en test, remarcando el mejor. Aguas abajo alimenta `src/fair_loss.py`, reutilizado por el tuner (NB 06) para la Pareto (E2).

## Decisiones a tomar antes de empezar

> Fichas de `docs/DECISIONES.md` para esta tarea. **Estado real** copiado tal cual.
> Las decisiones en **Propuesta**/**Abierta** se **validan con el grupo ANTES de
> implementar**: este notebook asume la *Propuesta* por defecto, pero es revisable.

| Decisión | Opciones | Estado |
|---|---|---|
| **D-2.1** · Medida de dependencia en la penalización | Pearson / Spearman / HSIC / dCor / MI — *Propuesta: HSIC o corr²* | Propuesta |
| **D-2.2** · Forma de combinar ajuste + penalización | `BCE + λ·D` (Lagrangiano) / escalarizadas | Propuesta |
| **D-2.3** · Métrica de equidad a reportar (S binaria) | group gap / tasas por grupo / CKA-HSIC residual — *Propuesta: group gap + tasas* | Propuesta |
| **D-2.4** · Métrica de precisión de la Pareto | AUC-ROC / accuracy / KS — *Propuesta: AUC-ROC* | Propuesta |
| **D-2.5** · Cómo pasar `S` a la loss | wrapper `q` (keras-fairkl) / concatenar `[y, S]` en `y_true` — *Propuesta: concatenar* | Propuesta |
| **D-2.6** · Dependencia sobre probabilidad o logit | probabilidad `ŷ` / logit — *Propuesta: probabilidad* | Propuesta |
| **D-2.7** · Tamaño de batch y σ del kernel | batch grande/pequeño; σ mediana global vs por batch | **Abierta** |

> **Premisa del EDA (por qué la FAIR loss es necesaria, no opcional).** Borrar `CODE_GENDER` **no** elimina el sesgo: el género se **filtra por `EXT_SOURCE_1`** (correlación **−0,31** con el género, F 0,546 / M 0,407), la variable más predictiva, y el group gap bruto **+3,14 pp** (M 10,14 % vs F 7,00 % de impago real) casi desaparece al controlar por ella. Por eso hay que **medir la dependencia contra `S` y penalizarla** ([`02-fair-loss.md`](../docs/teoria/02-fair-loss.md) §1).
>
> **D-2.7 queda Abierta**: el tamaño de batch y la σ del RBF (mediana global vs por batch) se **fijan tras pruebas** y se marcan *Revisar*. **Validar todas las fichas D-2.x con el grupo antes de codificar.**

In [1]:
# === Setup comun (notebooks de modelado 03-07) ===
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend unico para todo el grupo

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
RNG = 42
np.random.seed(RNG)
import random; random.seed(RNG)
try:
    import keras
    keras.utils.set_random_seed(RNG)
except Exception:
    pass

# Estilo heredado del EDA / preprocesado
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
COLOR_PAGA   = "#2c7fb8"   # TARGET=0  (paga)
COLOR_IMPAGA = "#d7301f"   # TARGET=1  (impaga)
COLOR_ACENTO = "#41ab5d"   # neutro

# Rutas estandar
PROC_DIR = Path("../data/processed")
FIG_DIR  = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR  = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- Especifico de la Tarea 2 (FAIR loss) ---
import keras
from sklearn.metrics import roc_auc_score
import sys; sys.path.append("..")          # para futuro: from src.fair_loss import fair_loss

## Carga del contrato `(X, y, s)`

Snippet estándar de convenciones (sección a): lee `metadata.json`, materializa `(X, y, s)` por split. **Aquí `s` (`CODE_GENDER`) SÍ se usa**: alimenta la **penalización de dependencia** de la FAIR loss (entra concatenado en `y_true`, **D-2.5**). Pero **`s` NUNCA es input de X** —X tiene exactamente las 13 features sin género, garantizado por el `assert` de no-fuga.

In [2]:
import json
from pathlib import Path
import pandas as pd

# --- Rutas y metadatos (fuente de verdad: metadata.json del preprocesado) ---
PROC_DIR   = Path("../data/processed")                       # relativo a notebooks/
META       = json.loads((PROC_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES_X = META["columns"]["features_X"]   # 13 features, en orden
SENSIBLE   = META["columns"]["sensible"]     # "CODE_GENDER"  (s)
TARGET     = META["columns"]["target"]       # "TARGET"       (y)

def cargar_split(nombre):
    """Devuelve (X, y, s) para 'train' | 'val' | 'test'.
    X = DataFrame solo con las 13 features (SIN genero).
    y = Series TARGET (1=impaga, 0=paga).  s = Series CODE_GENDER (M=1/F=0).
    """
    df = pd.read_parquet(PROC_DIR / f"{nombre}.parquet")
    X = df[FEATURES_X]          # input del modelo: el genero NUNCA entra aqui
    y = df[TARGET]
    s = df[SENSIBLE]
    assert SENSIBLE not in X.columns, "FUGA: el genero esta dentro de X"
    return X, y, s

# Materializar los tres cortes
X_train, y_train, s_train = cargar_split("train")
X_val,   y_val,   s_val   = cargar_split("val")
X_test,  y_test,  s_test  = cargar_split("test")

# Resumen de control
print(f"{'split':<7}{'X (filas, cols)':>20}{'y':>12}{'s':>12}{'tasa_impago':>14}")
for n, (X, y, s) in {"train": (X_train, y_train, s_train),
                     "val":   (X_val,   y_val,   s_val),
                     "test":  (X_test,  y_test,  s_test)}.items():
    print(f"{n:<7}{str(tuple(X.shape)):>20}{str(tuple(y.shape)):>12}"
          f"{str(tuple(s.shape)):>12}{y.mean():>14.4%}")

split       X (filas, cols)           y           s   tasa_impago
train          (215254, 13)   (215254,)   (215254,)       8.0728%
val             (46126, 13)    (46126,)    (46126,)       8.0735%
test            (46127, 13)    (46127,)    (46127,)       8.0734%


## Definición de la FAIR loss

Implementar `fair_loss(y_true, y_pred, lam)` en **`src/fair_loss.py`**: desempaqueta `y_real, s` de `y_true` (**D-2.5**), calcula `BCE(y_real, ŷ) + λ·D(ŷ, s)` sobre la **probabilidad** (**D-2.1**, **D-2.6**), con `D` = HSIC RBF (σ por heurística de la mediana) o `corr²`, todo en `keras.ops` para que sea diferenciable.

In [3]:
# TODO: fair_loss(y_true, y_pred, lam) en src/fair_loss.py
#   - desempaqueta  y_real, s = y_true[:, 0], y_true[:, 1]   (D-2.5)
#   - D = HSIC con kernel RBF (sigma = heuristica de la mediana) o corr^2  (D-2.1)
#   - devuelve  keras.ops.mean(BCE(y_real, y_pred)) + lam * D(y_pred, s)   (D-2.6: sobre la probabilidad)
#   - todo con keras.ops (diferenciable). D-2.7 (batch / sigma) queda Abierta -> fijar tras pruebas

## Modelo y cómo entra `S`

Mismo MLP sigmoide que la base (NB 03): X = 13 features (sin género) como input. **`s` entra solo por la loss**, empaquetando el objetivo como `y_true = np.column_stack([y, s])` (**D-2.5**); la red nunca ve el género.

In [4]:
# TODO: build_model() -> MLP Sequential Dense + Dropout + sigmoide (misma topologia que la base 03)
#   - input: X = 13 features (SIN genero)
#   - compile con la fair_loss(lam) y metrica de seguimiento
#   - s entra solo por la loss:  y_fair = np.column_stack([y, s])  (D-2.5); NUNCA como input de X

## Entrenamiento con barrido de `λ`

Reentrenar el modelo para cada `λ` del barrido y guardar el `history`; **`λ = 0` es la base de fairness** (modelo preciso sin penalización), y al subir `λ` las predicciones se vuelven más independientes del género ([`02-fair-loss.md`](../docs/teoria/02-fair-loss.md) §2).

In [5]:
# TODO: for lam in LAMBDAS (p.ej. [0, 0.5, 1, 2, 5, 8, 20]):
#   - construir y entrenar build_model() con fair_loss(lam) sobre y_fair = column_stack([y, s])
#   - guardar history[lam] para las curvas de loss (E4); lam=0 = base de fairness (sin penalizacion)

In [6]:
# TODO: figura E4 (convergencia) -> results/figures/05_fair__curva_loss.png

## Evaluación: AUC y group gap por `λ`

Para cada `λ` medir la **precisión** (AUC-ROC sobre `TARGET`, **D-2.4**) y la **equidad**: group gap `mean(ŷ|M) − mean(ŷ|F)` más las tasas por grupo (**D-2.3**). Cada fila es un punto del *trade-off*.

In [7]:
# TODO: por cada lam, sobre test (o val), medir:
#   - precision: AUC-ROC con roc_auc_score(y_test, y_pred)   (D-2.4)
#   - equidad: group gap = mean(y_pred[s==1]) - mean(y_pred[s==0]) + tasas por grupo   (D-2.3)
#   -> guardar tabla results/tables/05_fair__barrido_lambda.csv

In [8]:
# TODO: figura -> results/figures/05_fair__group_gap_vs_lambda.png
#   group gap y AUC frente a lambda; trazar la linea base de equidad +3,14 pp (EDA)

## Tabla E5 — Base (sin FAIR) vs mejor FAIR (test)

**Entregable central.** Elegir el mejor `λ` (codo de la frontera de Pareto), leer la métrica base del NB 03 y construir la tabla comparativa **base vs mejor FAIR** en test, **remarcando el mejor modelo**.

In [9]:
# TODO: tabla E5 (entregable central)
#   1) elegir mejor lam (codo de la Pareto: cae el group gap sin sacrificar demasiada AUC)
#
#   >>> DEPENDENCIA PENDIENTE (NB 03) <<<
#   2) LEER el baseline desde  results/tables/03_base__metricas_test.csv
#      (lo produce el notebook 03_modelo_base; SIN ese CSV esta celda NO puede completarse)
#
#   3) construir tabla [modelo Base (sin FAIR) | mejor FAIR] con AUC-ROC (D-2.4) y group gap (D-2.3) en test
#   4) -> results/tables/05_fair__base_vs_mejor_fair.csv  (remarcar el mejor modelo en test, E5)

## Entregables

- **E5 (principal)** · tabla **Base (sin FAIR) vs mejor FAIR** en test → `results/tables/05_fair__base_vs_mejor_fair.csv` (remarcando el mejor).
- **Barrido de `λ`** · `results/tables/05_fair__barrido_lambda.csv` (AUC y group gap por `λ`).
- **E4** · curva de loss del/los entrenamiento(s) final(es) → `results/figures/05_fair__curva_loss.png`.
- **Figura group gap vs `λ`** · `results/figures/05_fair__group_gap_vs_lambda.png` (con línea base +3,14 pp).
- **Presentación** · justificación de la **métrica de dependencia** elegida (HSIC/corr², **D-2.1**).

**Dependencias.** Aguas arriba: **NB 03** (baseline obligatorio para E5) y, opcional, **NB 04** (capa custom). Aguas abajo: define `λ` y `src/fair_loss.py`, que el **NB 06** (Keras Tuner) barre para la **curva de Pareto (E2)** reutilizando esta loss.